# HyperTools 2.0 dev/testing notebook

Interactive companion to the `dev-2.0` modernization branch (see `notes/hypertools_2.0_roadmap.md`).

**Purpose:** exercise EVERY public function across the full use-case matrix, visually verify output, and compare backends. Each section corresponds to one public API function. Run top-to-bottom in Jupyter, Colab, or Kaggle — the three environments we must support.

**Use-case matrix** (applied to each function where relevant):

| dimension | cases |
|-|-|
| input type | single ndarray, list of ndarrays, DataFrame, list of DataFrames, nested lists (→ MultiIndex), text |
| dimensionality | 2D, 3D, high-D (reduced), 1D edge case |
| missing data | none, NaNs (PPCA fill) |
| styling | fmt strings, hue/group, legend, labels, title, multilevel-index color/thickness/opacity |
| coloring | categorical labels, continuous values, mixture proportions, user matrices, multicolored lines |
| models | reduce (PCA/IncrementalPCA/TSNE/UMAP), cluster (KMeans/HDBSCAN), mixtures (GaussianMixture/BayesianGaussianMixture/LDA/NMF), align (hyper/SRM) |
| animation | static, animate=True, spin, sliding window |
| backend | matplotlib (default), plotly (interactive) — 2.0 feature |

In [ ]:
# Setup — when developing locally, install the branch in editable mode first:
#   pip install -e .
import sys, os, time
import numpy as np
import pandas as pd

t0 = time.time()
import hypertools as hyp
print(f'hypertools {hyp.__version__} imported in {time.time()-t0:.2f}s')  # 2.0 target: < 1s

ENV = 'colab' if 'google.colab' in sys.modules else ('kaggle' if os.path.exists('/kaggle') else 'local')
print('environment:', ENV)

In [ ]:
# Shared synthetic datasets (seeded for reproducibility)
rng = np.random.default_rng(42)
walk = np.cumsum(rng.standard_normal((200, 10)), axis=0)
walk2 = np.cumsum(np.random.default_rng(1).standard_normal((200, 10)), axis=0)
clusters = np.vstack([rng.standard_normal((60, 5)) + 6 * i for i in range(3)])
labels = [f'group{i}' for i in range(3) for _ in range(60)]
df = pd.DataFrame(walk, columns=[f'f{i}' for i in range(10)])
walk_missing = walk.copy(); walk_missing[rng.random(walk.shape) < 0.05] = np.nan

## 1. `hyp.plot` — core static cases

In [ ]:
geo = hyp.plot(walk)                      # 3D trajectory, single array
hyp.plot([walk, walk2])                   # list -> auto color per element
hyp.plot(clusters, 'o')                   # scatter via fmt string
hyp.plot(walk, '--', ndims=2)             # 2D + dashed linestyle (regression: linestyle parsing)
hyp.plot(df);                             # DataFrame input

## 2. `hyp.plot` — models: reduce / cluster / align

In [ ]:
hyp.plot(clusters, 'o', hue=labels, legend=True)
hyp.plot(clusters, 'o', cluster='KMeans', n_clusters=3)
hyp.plot(clusters, 'o', reduce='TSNE')
hyp.plot([walk, walk + 0.5], align='hyper')
hyp.plot(walk_missing);                   # PPCA missing-data interpolation

## 3. `hyp.plot` — animation

Known-broken on master: numpy>=2 Jupyter animations (#265), Colab `animate=True` (#235), figures in loops (#264). These cells are the acceptance tests for the 2.0 fixes.

In [ ]:
hyp.plot(walk, animate=True)              # sliding-window animation
hyp.plot(walk, animate='spin')            # camera spin
# regression #264: multiple figures in a loop
for seed in range(3):
    hyp.plot(np.cumsum(np.random.default_rng(seed).standard_normal((100, 5)), axis=0))

## 4. `hyp.plot` — 2.0 target APIs (will fail on master)

Acceptance targets for the features Jeremy flagged as critical carries: backend switching, multilevel-index/nested-list styling, mixture-model soft clustering, and the robust coloring pathway. Uncomment each block as it's implemented; every block also gets screenshot-harness cases.

In [ ]:
# 2.0 API (design per roadmap; uncomment as implemented):

# --- backend switching (policy approved: plotly only on Colab/Kaggle) ---
# hyp.plot(walk, backend='matplotlib')    # explicit default
# hyp.plot(walk, backend='plotly')        # interactive, works in Colab/Kaggle
# hyp.plot(walk, backend='auto')          # plotly on Colab/Kaggle, else matplotlib
# fig = hyp.plot(walk)                    # HyperToolsFigure wrapper
# fig.save('out.png'); fig.save('out.html')

# --- multilevel indices / nested lists (fork #14, #16) ---
# hyp.plot([[walk, walk2], [walk3]])      # nested list -> MultiIndex; color by outer
#                                         # level, thinner/fainter lines per deeper level

# --- mixture models: soft clustering (fork #10, #23) ---
# props = hyp.cluster(clusters, cluster='GaussianMixture', n_components=3)
#                                         # returns mixture PROPORTIONS, not labels
# hyp.plot(clusters, 'o', cluster='GaussianMixture', n_components=3)
#                                         # colors blended by membership weights
# hyp.plot(clusters, cluster='GaussianMixture', n_components=3)
#                                         # line mode + soft assignments (#23 regression)

# --- robust coloring via mat2colors/vals2colors (fork #11, #24, #32) ---
# hyp.plot(walk, hue=np.arange(len(walk)))            # continuous values -> palette
# hyp.plot(walk, hue=weights_matrix)                  # arbitrary matrix -> reduce -> color
# hyp.plot([df1, df2], 'k.')                          # fmt+color on list input (#24 crash)

## 5. `hyp.reduce`

In [ ]:
print(hyp.reduce(walk, ndims=3).shape)
print(hyp.reduce([walk, walk2], ndims=2)[0].shape)
print(hyp.reduce(walk, reduce='IncrementalPCA', ndims=4).shape)

## 6. `hyp.align`

In [ ]:
aligned = hyp.align([walk, walk + 0.5])
print([a.shape for a in aligned])
aligned_srm = hyp.align([walk, walk + 0.5], align='SRM')
print([a.shape for a in aligned_srm])

## 7. `hyp.cluster`, `hyp.normalize`, `hyp.analyze`, `hyp.describe`

In [ ]:
print(np.unique(hyp.cluster(clusters, n_clusters=3)))
print(hyp.normalize(walk).mean(axis=0).round(3)[:5])
print(hyp.analyze(walk, ndims=3, normalize='within').shape)
hyp.describe(walk);

## 8. `hyp.load` + text input (network required)

In [ ]:
geo = hyp.load('weights_sample')
geo.plot()
hyp.plot(['the quick brown fox', 'jumped over the lazy dog',
          'machine learning is fun', 'high dimensional data'], 'o');

## 9. Screenshot capture (headless verification)

For the systematic screenshot matrix, run:
```bash
python scripts/generate_baseline_screenshots.py
```
and review PNGs under `tests/screenshots/`.